<p><font size="6"><b> Visualizing spatial data with Python: GeoPandas</b></font></p>


> *DS Python for GIS and Geoscience*  
> *November, 2026*
>
> *© 2026, Joris Van den Bossche and Stijn Van Hoey. Licensed under [CC BY 4.0 Creative Commons](https://creativecommons.org/licenses/by/4.0/)*

---

In [ ]:
import pandas as pd
import geopandas

import matplotlib.pyplot as plt

In [ ]:
countries = geopandas.read_file("data/ne_110m_admin_0_countries.zip")
countries = countries[countries['continent'] != "Antarctica"]
cities = geopandas.read_file("data/ne_110m_populated_places.zip")
rivers = geopandas.read_file("data/ne_50m_rivers_lake_centerlines.zip")

## GeoPandas visualization functionality

GeoPandas itself provides some visualization functionality, and together with matplotlib for further customization, you can already get decent results for visualizing vector data.

#### Basic plot

In [ ]:
countries.plot()

<div class="alert alert-info" style="font-size:120%">

**NOTE**: <br>

Making a quick interactive plot is also available as the `.explore()` method on a GeoDataFrame or GeoSeries, using the Folium package. See the [visualization-04-interactive notebook](./visualization-04-interactive.ipynb) for more information on interactive plotting packages or check the [GeoPandas documentation](https://geopandas.org/en/stable/docs/user_guide/interactive_mapping.html) for more examples.

</div>

In [ ]:
countries.explore()

#### Adjusting the figure size

In [ ]:
countries.plot(figsize=(15, 6))

#### Removing the box / x and y coordinate labels

In [ ]:
ax = countries.plot(figsize=(15, 6))
ax.set_axis_off()

#### Coloring based on column values

Let's first create a new column with the GDP per capita:

In [ ]:
countries = countries[(countries['pop_est'] >0 ) & (countries['name'] != "Antarctica")]

In [ ]:
countries['gdp_per_cap'] = countries['gdp_md_est'] / countries['pop_est'] * 100

and now we can use this column to color the polygons:

In [ ]:
ax = countries.plot(figsize=(15, 6), column='gdp_per_cap', legend=True)
ax.set_axis_off()

In [ ]:
# use the "compressed" layout of matplotlib to get better default sizing of the colorbar
fig, ax = plt.subplots(figsize=(15, 6), layout="compressed")
ax = countries.plot(ax=ax, column='gdp_per_cap', legend=True)
ax.set_axis_off()

Using a classification scheme to bin the values (using [`mapclassify`](https://pysal.org/mapclassify/)):

In [ ]:
ax = countries.plot(figsize=(15, 6), column='gdp_per_cap', scheme='quantiles', legend=True)
ax.set_axis_off()

In [ ]:
# same, but keeping a colorbar for the legend
ax = countries.plot(figsize=(15, 6), column='gdp_per_cap', scheme='quantiles', legend=True, legend_kwds={"colorbar": True})
ax.set_axis_off()

#### Combining different dataframes on a single plot

The `.plot` method returns a matplotlib Axes object, which can then be re-used to add additional layers to that plot with the `ax=` keyword:

In [ ]:
ax = countries.plot(figsize=(15, 6))
cities.plot(ax=ax, color='red', markersize=10)
ax.set_axis_off()

In [ ]:
ax = countries.plot(edgecolor='k', facecolor='none', figsize=(8, 8))
rivers.plot(ax=ax)
cities.plot(ax=ax, color='C1')
ax.set(xlim=(-20, 60), ylim=(-40, 40))

## Adding a background map with contextily

The [contextily](https://contextily.readthedocs.io/en/latest/) package allows to easily add a web-tile based background (basemap) to your GeoPandas plots.

Starting with `geopandas` >= 1.2, this is available as a `tiles=True` shortcut in the `plot()` method:

In [ ]:
# selecting the cities in Europe
cities_europe = cities[cities.within(countries[countries['continent'] == 'Europe'].union_all())]

In [ ]:
cities_europe.plot()

In [ ]:
cities_europe.plot(figsize=(10, 6), tiles=True)

The tiles are in the WebMercator projection (EPSG:3857), and thus to avoid warping of the tile images, it can be advisable to reproject your data to WebMercator.

The following example also shows how to add the basemap yourself using `contextily`:

In [ ]:
# converting to WebMercator
cities_europe2 = cities_europe.to_crs(epsg=3857)

In [ ]:
import contextily

In [ ]:
# import matplotlib
# matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
ax = cities_europe2.plot(figsize=(10, 6))
contextily.add_basemap(ax)

In [ ]:
# specifying a different tile provider\
ax = cities_europe2.plot(figsize=(10, 6))
contextily.add_basemap(ax, source=contextily.providers.NASAGIBS.BlueMarble)

See the [contextily documentation](https://contextily.readthedocs.io/en/latest/index.html) for more information how to customize this (e.g. the zoom level of the tiles), and the [xyzservices documentation](https://xyzservices.readthedocs.io/en/stable/index.html) for an overview of all available (built-in) tile providers.

## Projection-aware maps with Cartopy

Cartopy is the base matplotlib cartographic library, and it can be used to create projection-aware maps (http://scitools.org.uk/cartopy/docs/latest/index.html).

The following example is taken from the docs: http://geopandas.readthedocs.io/en/latest/gallery/cartopy_convert.html#sphx-glr-gallery-cartopy-convert-py

In [ ]:
from cartopy import crs as ccrs

In [ ]:
crs.to_wkt()

In [ ]:
# Define the CartoPy CRS object.
crs = ccrs.AlbersEqualArea()

# This crs object is compatible with GeoPandas, and will be recognized by its to_crs() method
countries_ae = countries.to_crs(crs)

In [ ]:
# Here's what the plot looks like in GeoPandas
countries_ae.plot()

In [ ]:
# Here's what the plot looks like when plotting with cartopy
fig, ax = plt.subplots(subplot_kw={'projection': crs})
ax.add_geometries(countries_ae['geometry'], crs=crs)
ax.gridlines()

In [ ]:
# Here's what the plot looks like when plotting with cartopy and geopandas combined
fig, ax = plt.subplots(subplot_kw={'projection': crs})
countries_ae['geometry'].plot(ax=ax)
ax.set_global()
ax.gridlines()

**For more on cartopy**, see the [visualization-03-cartopy.ipynb](visualization-03-cartopy.ipynb) notebook.